For the Eyetracking Experiment the design matrix needs to:
- have balanced L/R target representation per image


In [1]:
from IPython.display import Markdown, display

display(Markdown("""
| | **Neutral L** | **Neutral R** | **Expected L** | **Expected R** | **Unexpected L** | **Unexpected R** |
|---|---|---|---|---|---|---|
| **Short** | 11–18 | 111–118 | 41–48 | 141–148 | 71–78 | 171–178 |
| **Long** | 21–28 | 121–128 | 51–58 | 151–158 | 81–88 | 181–188 |
"""))



| | **Neutral L** | **Neutral R** | **Expected L** | **Expected R** | **Unexpected L** | **Unexpected R** |
|---|---|---|---|---|---|---|
| **Short** | 11–18 | 111–118 | 41–48 | 141–148 | 71–78 | 171–178 |
| **Long** | 21–28 | 121–128 | 51–58 | 151–158 | 81–88 | 181–188 |


In [1]:

import numpy as np
import random
import os
import pandas as pd 
from collections import Counter

#=====================================================
# Trial Functions
# =====================================================

def create_cue_dynam(highProb=0.7, lowProb=0.3, neutral=1.0, trials_per_cue=40, trial_per_neutral=32):
    
    cue_data = {"cue_names": [".\\cues\\Sea_Animal.png",  ".\\cues\\Water_Vehicle.png",  ".\\cues\\Neutral.png"],
                "cue_highProb_cats": [["dolphin", "whale"], ["speedboat", "submarine"], 
                                    ["dolphin", "whale", "speedboat", "submarine"]],
                
                "cue_lowProb_cats": [["speedboat", "submarine"], ["dolphin", "whale"],
                                    ["dolphin", "whale", "speedboat", "submarine"]]}

    cue_data["cue_highProb"] = []
    cue_data["cue_lowProb"] = []
    
    for cue_id, cue in enumerate(cue_data["cue_names"]):
        if cue != ".\\cues\\Neutral.png":
            cue_data["cue_highProb"].append([np.round(highProb / len(cue_data["cue_highProb_cats"][cue_id]), 2)] * len(cue_data["cue_highProb_cats"][cue_id]))
            cue_data["cue_lowProb"].append([np.round(lowProb / len(cue_data["cue_lowProb_cats"][cue_id]), 2)] * len(cue_data["cue_lowProb_cats"][cue_id]))
        
        else:
            cue_data["cue_highProb"].append([np.round(neutral / len(cue_data["cue_highProb_cats"][cue_id]), 3)] * len(cue_data["cue_highProb_cats"][cue_id]))
            cue_data["cue_lowProb"].append([np.round(neutral / len(cue_data["cue_lowProb_cats"][cue_id]), 3)] * len(cue_data["cue_highProb_cats"][cue_id]))
            
    cue_data["high_prob_trials"] = [
    np.array(x) * (trial_per_neutral if cue_data["cue_names"][idx] == ".\\cues\\Neutral.png" else trials_per_cue)
    for idx, x in enumerate(cue_data["cue_highProb"])]
    
    cue_data["low_prob_trials"] =  [
    np.array(x) * (trial_per_neutral if cue_data["cue_names"][idx] == ".\\cues\\Neutral.png" else trials_per_cue)
    for idx, x in enumerate(cue_data["cue_lowProb"])]
    
    return cue_data

def assign_trigger(row, late=0.100):

    # ---- MASK ----
    if row['mask_ISI'] == 0.0165:
        mask_code = 10
    elif row['mask_ISI'] == late:
        mask_code = 20
    else:
        raise ValueError("Unknown mask type")

    # ---- SIDE ----
    if row['target_loc'] == 'L':
        side_code = 0
    elif row['target_loc'] == 'R':
        side_code = 100
    else:
        raise ValueError("Unknown side")

    # ---- EXPECTATION ----
    if row['expectation'] == 'neutral':
        exp_code = 0
    elif row['expectation'] == 'expected':
        exp_code = 30
    elif row['expectation'] == 'unexpected':
        exp_code = 60
    else:
        raise ValueError("Unknown expectation")

    # ---- IMAGE ----
    image_code = row['image_index']  # 1–8

    return mask_code + side_code + exp_code + image_code

def create_changes(list_length, prec):
    # Number of instances to set as True
    num_true = int(list_length * prec)

    # Create lists of False values
    catch = [False] * list_length

    # Randomly select indices for fixing and imaging channels
    catch_indices = random.sample(range(list_length), num_true)

    for idx in catch_indices:
        catch[idx] = True
    
    return np.array(catch)

def allocate_catch_trials(N, p=0.3, k=3, alpha=0.66):
    C = int(N * p)
    
    # ensure divisible by (k-1)*2 if needed
    while C % (k-1) != 0:
        C -= 1
    
    main = int(round(alpha * C))
    
    # enforce even split
    remainder = C - main
    other = remainder // (k - 1)
    
    # adjust if rounding broke divisibility
    main = C - other * (k - 1)
    other = [other] * (k - 1)
    
    return main, other[0], other[1]

def assign_catch_trials(df, rng=None, p=0.25, k=3, alpha=0.5, cond="identity_catch"):
    """
    Takes your existing 224-trial dataframe and:
      1. Flags 56 rows as catch trials (24 neutral, 14 expected, 14 unexpected),
         balanced across mask_ISI within each expectation condition.
      2. Reorders the sequence so catches are spaced min=2, max=7-8, mean~4 apart.
    
    Adds an cond boolean column.
    Returns a reordered dataframe (reset index).
    """
    if rng is None:
        rng = random.Random()

    df = df.copy()
    df[cond] = False

    # --- 1. Flag catch trials ---
    neut, exp, unexp = allocate_catch_trials(len(df), p, k, alpha)
    catch_counts = {'neutral': neut, 'expected': exp, 'unexpected': unexp}

    for condition, n_catches in catch_counts.items():
        cond_idx = df[df['expectation'] == condition].index.tolist()
        assert len(cond_idx) >= n_catches, \
            f"Not enough {condition} trials: need {n_catches}, have {len(cond_idx)}"

        # Balance across mask_ISI: half from 0.0165, half from 0.100
        half = n_catches // 2  # both 52 and 14 are even, so no remainder

        for isi_val, n in [(0.0165, half), (0.100, half)]:
            isi_idx = df.loc[cond_idx][df.loc[cond_idx, 'mask_ISI'] == isi_val].index.tolist()
            assert len(isi_idx) >= n, \
                f"Not enough {condition}/ISI={isi_val} trials: need {n}, have {len(isi_idx)}"
            chosen = rng.sample(isi_idx, n)
            df.loc[chosen, cond] = True

    # --- 2. Separate catches and non-catches ---
    catch_df    = df[df[cond]].sample(frac=1, random_state=rng.randint(0, 99999)).reset_index(drop=True)
    noncatch_df = df[~df[cond]].sample(frac=1, random_state=rng.randint(0, 99999)).reset_index(drop=True)

    n_catches  = len(catch_df)    # 80
    n_noncatch = len(noncatch_df) # 240

    # --- 3. Sample inter-catch gaps ---
    # gap = number of non-catch trials BEFORE each catch
    # gap in [1, 7] → total catch-to-catch distance of [2, 8], mean ~4
    gaps = _sample_gaps(
        n_catches  = n_catches,
        n_noncatch = n_noncatch,
        gap_min    = 1,
        gap_max    = 7,
        gap_mean   = 3.0,  # 3 non-catches between → distance of 4
        rng        = rng,
    )

    # --- 4. Interleave into final sequence ---
    sequence_rows = []
    nc_pointer = 0

    for i, gap in enumerate(gaps):
        # Insert `gap` non-catch trials
        for _ in range(gap):
            sequence_rows.append(noncatch_df.iloc[nc_pointer])
            nc_pointer += 1
        # Insert catch trial
        sequence_rows.append(catch_df.iloc[i])

    # Append any leftover non-catch trials at the end
    while nc_pointer < n_noncatch:
        sequence_rows.append(noncatch_df.iloc[nc_pointer])
        nc_pointer += 1

    result = pd.DataFrame(sequence_rows).reset_index(drop=True)
    return result

def _sample_gaps(n_catches, n_noncatch, gap_min, gap_max, gap_mean, rng):
    """
    Returns a list of n_catches integers in [gap_min, gap_max]
    that sum exactly to n_noncatch, distributed around gap_mean.
    """
    # Sanity check: is the target sum achievable?
    assert gap_min * n_catches <= n_noncatch <= gap_max * n_catches, (
        f"Cannot distribute {n_noncatch} non-catch trials across {n_catches} gaps "
        f"with min={gap_min}, max={gap_max}. "
        f"Feasible range: [{gap_min * n_catches}, {gap_max * n_catches}]"
    )

    for _ in range(50_000):
        gaps = [
            max(gap_min, min(gap_max, round(rng.triangular(gap_min, gap_max, gap_mean))))
            for _ in range(n_catches)
        ]
        diff = sum(gaps) - n_noncatch

        # Nudge gaps up or down to hit the exact sum
        for _ in range(5_000):
            if diff == 0:
                break
            idx = rng.randrange(n_catches)
            if diff > 0 and gaps[idx] > gap_min:
                gaps[idx] -= 1
                diff -= 1
            elif diff < 0 and gaps[idx] < gap_max:
                gaps[idx] += 1
                diff += 1

        if diff == 0:
            return gaps

    raise RuntimeError("Failed to converge on valid gap distribution.")

def validate_sequence(df, catch_cond="identity_catch"):
    catches = df[df[catch_cond]]

    print(f"Total:     {len(df)}")
    print(f"Catch {catch_cond}:     {len(catches)}  ({len(catches)/len(df)*100:.1f}%)")
    print(f"Non-catch: {len(df) - len(catches)}")

    print(f"\nCatch breakdown by expectation:")
    for cond, grp in catches.groupby('expectation'):
        c017 = (grp['mask_ISI'] == 0.0165).sum()
        c100 = (grp['mask_ISI'] == 0.100).sum()
        print(f"  {cond:11s}: total={len(grp)}  ISI=0.0165: {c017}  ISI=0.100: {c100}")

    catch_pos = df.index[df[catch_cond]].tolist()
    distances = [catch_pos[i+1] - catch_pos[i] for i in range(len(catch_pos) - 1)]

    print(f"\nCatch-to-catch distances:")
    print(f"  min={min(distances)}  max={max(distances)}  mean={np.mean(distances):.2f}")
    print(f"  distribution: {dict(sorted(Counter(distances).items()))}")

def build_constrained_order(df, rng=None, max_unexpected_run=1, max_attempts=1000):
    
    if rng is None:
        rng = random.Random()

    for _ in range(max_attempts):
        remaining = df.copy().reset_index(drop=True)
        ordered_rows = []
        last_target = None
        unexpected_run = 0
        failed = False

        while len(remaining) > 0:

            # Build valid candidates mask
            valid_mask = np.ones(len(remaining), dtype=bool)

            # Rule 1 — no same target twice in a row
            if last_target is not None:
                valid_mask &= (remaining["target"].astype(str) != str(last_target))

            # Rule 2 — max consecutive unexpected trials
            if unexpected_run >= max_unexpected_run:
                valid_mask &= (remaining["expectation"].astype(str) != "unexpected")

            # Integer positions of valid rows within remaining
            valid_positions = np.where(valid_mask)[0]

            if len(valid_positions) == 0:
                failed = True
                break

            # Pick a random valid position
            chosen_pos = valid_positions[rng.randrange(len(valid_positions))]
            row = remaining.iloc[chosen_pos]

            ordered_rows.append(row)

            # Update state
            last_target = str(row["target"])
            unexpected_run = unexpected_run + 1 if str(row["expectation"]) == "unexpected" else 0

            # Drop by integer position and reset index
            remaining = remaining.drop(remaining.index[chosen_pos]).reset_index(drop=True)

        if not failed:
            return pd.DataFrame(ordered_rows).reset_index(drop=True)

    raise RuntimeError(
        f"Could not build a valid trial order after {max_attempts} attempts. "
        "Consider relaxing the constraints."
    )

def get_second_selection(row):
    group1 = {0, 1, 14, 15}
    group2 = {6, 7, 10, 11}

    if row["target_id"] in group1 and row["distractor_selection_match_id"] in group1:
        return [(6, 7), (10, 11)][np.random.randint(2)]

    elif row["target_id"] in group2 and row["distractor_selection_match_id"] in group2:
        return [(0, 1), (14, 15)][np.random.randint(2)]

    return (np.nan, np.nan)

def assign_locations(row):
    # randomly choose target location
    target_loc = np.random.choice(["up", "down", "left", "right"])

    match_map = {
        "up": "left",
        "left": "up",
        "down": "right",
        "right": "down",
    }

    distractor_match_loc = match_map[target_loc]

    remaining = list(
        {"up", "down", "left", "right"}
        - {target_loc, distractor_match_loc}
    )

    np.random.shuffle(remaining)

    return pd.Series({
        "target_selection_loc": target_loc,
        "distractor_match_selection_loc": distractor_match_loc,
        "distractor_2_selection_loc": remaining[0],
        "distractor_2_match_selection_loc": remaining[1],
    })

def create_block_trials(stim_path, cue_data, random_seed, long_isi=0.1, identity_catch=1.0, location_catch=0.0, k=3, alpha=0.5):
    rng = random.Random(random_seed)
    long_isi = 0.1
    categories = os.listdir(stim_path)
    stimuli = []
    for cat in categories:
        cat_path = os.path.join(stim_path, cat)
        files = os.listdir(cat_path)
        stimuli.extend([f".\\stimuli\\{cat}\\{x}" for x in files])

    stimuli = np.array(stimuli)
    stimuli = stimuli[np.argsort(stimuli)]

    data = {
        "target_id": [],
        "distractor_id": [],
        "distractor_selection_match_id": [],
        "target": [],
        "distractor": [],
        "distractor_selection_match": [],
        "expectation": [],
        "mask_ISI": [],
        "cue": [],
        "target_name": [],
        "target_cat": [],
        "target_loc": []
    }

    mask_type = [0.0165, long_isi]
    distractors = stimuli[np.char.count(stimuli, "mask") > 0]
    distractor_ids = np.arange(len(stimuli))[np.isin(stimuli, distractors)]
    local_distractor_ids = np.arange(0, len(distractors))

    mapping = {
        0: 1,
        1: 2,
        14: 3,
        15: 4,
        6: 5,
        7: 6,
        10: 7,
        11: 8
    }

    def add_trials(targets, target_ids, n_trials, mask, cue, expectation_label):
        """
        Add n_trials rows for a given set of targets, guaranteeing all
        data-dict lists grow by exactly n_trials entries.

        The target images are cycled so each gets as equal exposure as
        possible, independent of whether n_trials is divisible by the
        number of images or by 2.
        """
        n_targets = len(targets)
        if n_targets == 0 or n_trials == 0:
            return

        target_names = [x.split("\\")[-1] for x in targets]
        target_categories = [x.split("_")[0] for x in target_names]

        # Build per-trial target assignments by cycling through images
        # so every image appears as evenly as possible.
        repeats = [n_trials // n_targets + (1 if i < n_trials % n_targets else 0)
                   for i in range(n_targets)]

        trial_target_ids   = []
        trial_targets      = []
        trial_dist_sel_ids = []
        trial_dist_sels    = []
        trial_names        = []
        trial_cats         = []

        for i, r in enumerate(repeats):
            trial_target_ids.extend([target_ids[i]] * r)
            trial_targets.extend([targets[i]] * r)
            
            # paired distractor-selection is the "other" image (flipped)
            paired_idx = (i + 1) % n_targets
            trial_dist_sel_ids.extend([target_ids[paired_idx]] * r)
            trial_dist_sels.extend([targets[paired_idx]] * r)
            trial_names.extend([target_names[i]] * r)
            trial_cats.extend([target_categories[i]] * r)

        # Interleave L/R target locations as evenly as possible
        locs = (["L", "R"] * (n_trials // 2 + 1))[:n_trials]

        random_distractors = np.random.choice(local_distractor_ids, n_trials)

        data["target_id"].extend(trial_target_ids)
        data["target"].extend(trial_targets)
        
        data["distractor_selection_match_id"].extend(trial_dist_sel_ids)
        data["distractor_selection_match"].extend(trial_dist_sels)
        
        data["target_loc"].extend(locs)
        data["target_name"].extend(trial_names)
        data["target_cat"].extend(trial_cats)
        data["distractor_id"].extend([distractor_ids[x] for x in random_distractors])
        data["distractor"].extend([distractors[x] for x in random_distractors])
        data["expectation"].extend([expectation_label] * n_trials)
        data["mask_ISI"].extend([mask] * n_trials)
        data["cue"].extend([cue] * n_trials)

    for cue_id, cue in enumerate(cue_data["cue_names"]):
        high_cats = np.array(cue_data["cue_highProb_cats"][cue_id])
        low_cats  = np.array(cue_data["cue_lowProb_cats"][cue_id])
        is_neutral = (cue == '.\\cues\\Neutral.png')

        # ---- LOW-PROBABILITY / UNEXPECTED trials (non-neutral cues only) ----
        if not is_neutral:
            for i, l_cat in enumerate(low_cats):
                l_cat_stim = stimuli[np.char.count(stimuli, l_cat) > 0]
                targets    = l_cat_stim[np.char.count(l_cat_stim, "mask") == 0]
                target_ids = np.arange(len(stimuli))[np.isin(stimuli, targets)]

                for mask in mask_type:
                    # Round to nearest even number so L/R split is always exact
                    n_trials = int(cue_data["low_prob_trials"][cue_id][i])
                    if n_trials % 2 != 0:
                        n_trials += 1
                    add_trials(targets, target_ids, n_trials, mask, cue, "unexpected")

        # ---- HIGH-PROBABILITY / EXPECTED (or NEUTRAL) trials ----
        for i, h_cat in enumerate(high_cats):
            h_cat_stim = stimuli[np.char.count(stimuli, h_cat) > 0]
            targets    = h_cat_stim[np.char.count(h_cat_stim, "mask") == 0]
            target_ids = np.arange(len(stimuli))[np.isin(stimuli, targets)]

            expectation_label = "neutral" if is_neutral else "expected"

            for mask in mask_type:
                n_trials = int(cue_data["high_prob_trials"][cue_id][i])
                if n_trials % 2 != 0:
                    n_trials += 1
                add_trials(targets, target_ids, n_trials, mask, cue, expectation_label)

    # ---- Derived columns (computed after all rows are collected) ----
    data["distractor_loc"] = ["R" if x == "L" else "L" for x in data["target_loc"]]
    df = pd.DataFrame(data)
    df['image_index'] = df['target_id'].map(mapping)
    df['trigger']     = df.apply(assign_trigger, axis=1)
    df = build_constrained_order(df, rng=rng)

    df[
    ["distractor_selection_2_id", "distractor_selection_2_match_id"]
    ] = df.apply(get_second_selection, axis=1, result_type="expand")

    df["distractor_selection_2"] = (
        df["distractor_selection_2_id"]
        .map(lambda x: stimuli[int(x)].split("\\")[-1] if not pd.isna(x) else np.nan)
    )

    df["distractor_selection_2_match"] = (
        df["distractor_selection_2_match_id"]
        .map(lambda x: stimuli[int(x)].split("\\")[-1] if not pd.isna(x) else np.nan)
    )


    df[[
            "target_selection_loc",
            "distractor_match_selection_loc",
            "distractor_2_selection_loc",
            "distractor_2_match_selection_loc",
        ]] = df.apply(assign_locations, axis=1)

    if identity_catch == 1.0 or identity_catch == 0.0:
        df["identity_catch"] = create_changes(len(df), identity_catch)
    else:
        df = assign_catch_trials(df, rng=rng, p=identity_catch, k=k, alpha=alpha, cond="identity_catch")
        validate_sequence(df, catch_cond="identity_catch")

    if location_catch == 1.0 or location_catch == 0.0:
        df["location_catch"] = create_changes(len(df), location_catch)
    else:
        df = assign_catch_trials(df, rng=rng, p=location_catch, k=k, alpha=alpha, cond="location_catch")
        validate_sequence(df, catch_cond="location_catch")

    return df, stimuli


In [2]:

cue_data = create_cue_dynam(highProb=0.7, lowProb=0.3, neutral=1.0, trials_per_cue=40)
stim_path = "/projects/crunchie/boyanova/EEG_Things/Mask_ExpAtt_EEG/01_Eyelink_Task_eyeResponse/stimuli"
trials, stimss = create_block_trials(stim_path, cue_data, random_seed=19, identity_catch=0.25, location_catch=1.0)

Total:     224
Catch identity_catch:     56  (25.0%)
Non-catch: 168

Catch breakdown by expectation:
  expected   : total=14  ISI=0.0165: 7  ISI=0.100: 7
  neutral    : total=28  ISI=0.0165: 14  ISI=0.100: 14
  unexpected : total=14  ISI=0.0165: 7  ISI=0.100: 7

Catch-to-catch distances:
  min=2  max=8  mean=3.98
  distribution: {2: 7, 3: 12, 4: 21, 5: 9, 6: 3, 7: 2, 8: 1}


In [1]:
import os
from PIL import Image

stim_dir = "/projects/crunchie/boyanova/EEG_Things/Mask_ExpAtt_EEG/01_Eyelink_Task/stimuli"
categories = os.listdir(stim_dir)
target_size = (500, 500)

# Image extensions to process
valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")

for c in categories:
    root_dir = os.path.join(stim_dir, c)
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.lower().endswith(valid_exts):
                img_path = os.path.join(root, file)

                try:
                    with Image.open(img_path) as img:
                        img = img.convert("RGB")  # safe for consistency
                        img_resized = img.resize(target_size, Image.LANCZOS)
                        img_resized.save(img_path)

                except Exception as e:
                    print(f"Failed to process {img_path}: {e}")


In [1]:
ecc             = 195
pos_left        = (-ecc, 0)
pos_right       = ( ecc, 0)

import math 
    
def pixels_to_degrees(pixels, distance, screen_width, resolution_width):
    """
    Convert pixels to degrees of visual angle.

    Parameters:
        pixels (int): Number of pixels to convert.
        distance (float): Distance from the observer to the screen (same units as screen_width).
        screen_width (float): Physical width of the screen (same units as distance).
        resolution_width (int): Horizontal resolution of the screen (in pixels).

    Returns:
        float: Visual angle in degrees.
    """
    # Physical size of a single pixel
    pixel_size = screen_width / resolution_width

    # Physical size of the object (in the same units as screen_width)
    object_size = pixel_size * pixels

    # Calculate visual angle using the formula
    visual_angle = 2 * math.degrees(math.atan((object_size / 2) / distance))

    return visual_angle

In [2]:
screenWidth = 41.0      # cm
screenHeight = 31.0     # cm
resX = 1600
resY = 1200
dist2Screen = 57    

pixels_to_degrees(195, dist2Screen, screenWidth, resX)

5.01959134564017